In [2]:
import os  # Provides functions for working with the operating system (files, directories, paths)
import numpy as np   # Used for numerical operations and handling arrays
import pandas as pd  # Used for data manipulation and analysis in tabular form
import librosa   # A library for audio and music processing (feature extraction, loading sounds)
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical


In [3]:
dataset_path = r"D:\TalentPrism\Gender Detection\gender-detection\Voice Detection\archive\data"  # Define the root path of the dataset containing different labeled folders

# Initialize two empty lists to store file paths and their corresponding labels
paths = []
labels = []

# Loop through each folder (label) inside the dataset directory
for label in os.listdir(dataset_path):
    label_folder = os.path.join(dataset_path, label)
    if os.path.isdir(label_folder):
        for file in os.listdir(label_folder):
            if file.endswith(".wav"):
                paths.append(os.path.join(label_folder, file))
                labels.append(label.lower())

print(f"Total samples: {len(paths)}")
print(f"Labels: {set(labels)}")    # Print the unique set of labels found in the dataset

Total samples: 16148
Labels: {'male', 'female'}


### Why MFCC?

They reduce high-dimensional audio data into a smaller feature vector while preserving information useful for distinguishing different sounds

In [23]:
import librosa
import numpy as np

def extract_features(file_path, n_mfcc=40):
    try:
        y, sr = librosa.load(file_path, sr=16000)

        # MFCC
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean = np.mean(mfcc.T, axis=0)

        # Pitch (F0)
        pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
        pitch_values = pitches[pitches > 0]
        pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0

        # Combine MFCC + Pitch
        features = np.hstack([mfcc_mean, pitch_mean])

        return features

    except Exception as e:
        print("Error:", file_path, e)
        return None


In [24]:
X = []

for path in paths:
    feat = extract_features(path)
    if feat is not None:
        X.append(feat)

X = np.array(X)
y = np.array(y[:len(X)])

print(X.shape)  # (samples, 41)


(16148, 41)


In [25]:
from sklearn.preprocessing import LabelEncoder

# Create a LabelEncoder object to convert string labels into numeric labels
le = LabelEncoder()
# Fit the encoder on y (learn all unique classes) and transform them into integers
y = le.fit_transform(y)  # male=1, female=0
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Label mapping: {0: 0, 1: 1}


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [28]:
# Build a Sequential neural network model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model: define optimizer, loss function, and evaluation metrics
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display a summary of the model architecture (layers, shapes, parameters)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                2688      
                                                                 
 dropout_2 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dropout_3 (Dropout)         (None, 32)                0         
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 


Total params: 4801 (18.75 KB)
Trainable params: 4801 (18.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [29]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights to handle imbalance
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
# Create a dictionary mapping: {class_label: weight}
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)


Class weights: {0: 1.3998699609882965, 1: 0.7778179190751445}


In [30]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,    # Number of passes over the training data
    batch_size=32,
    shuffle=True,
    class_weight=class_weights  # Apply class weights to handle class imbalance
)


Epoch 1/30
323/323 [==============================] - 3s 5ms/step - loss: 0.0852 - accuracy: 0.9718 - val_loss: 0.0078 - val_accuracy: 0.9981
Epoch 2/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0071 - accuracy: 0.9984 - val_loss: 0.0025 - val_accuracy: 0.9996
Epoch 3/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0027 - accuracy: 0.9996 - val_loss: 0.0016 - val_accuracy: 0.9996
Epoch 4/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0026 - accuracy: 0.9995 - val_loss: 5.9294e-04 - val_accuracy: 1.0000
Epoch 5/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0013 - accuracy: 0.9997 - val_loss: 3.7952e-04 - val_accuracy: 1.0000
Epoch 6/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0016 - accuracy: 0.9995 - val_loss: 5.3786e-04 - val_accuracy: 0.9996
Epoch 7/30
323/323 [==============================] - 1s 4ms/step - loss: 5.7534e-04 - accuracy: 0.9998 - val_loss: 8.3316e-04 - val_acc

In [31]:
# Evaluate the trained model on unseen test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

101/101 [==============================] - 0s 3ms/step - loss: 0.0025 - accuracy: 0.9991
Test Accuracy: 99.91%


# Prediction of Gender

In [46]:
def predict_gender(file_path):
    features = extract_features(file_path)   # ✅ SAME function as training

    if features is None:
        return "Error processing audio"

    features = scaler.transform([features])  # shape (1, 41)
    prob = model.predict(features, verbose=0)[0][0]

    print("Probability:", prob)
    return "Male" if prob >= 0.5 else "Female"


In [52]:
print(predict_gender(r"D:\TalentPrism\Gender Detection\gender-detection\Voice Detection\archive\testaudio\test5.wav"))

Probability: 1.0
Male
